# Predict with the deployed model

- **Author**: senkin.zhan@datarobot.com
- **Accelerator**: Deep learning for molecular SMILES: train and deploy on DataRobot — see [`README.md`](README.md)

Scores SMILES through the **DataRobot deployment** instead of loading `model/smiles_model.pth` locally,
so this notebook needs no torch / rdkit — only the DataRobot SDK.

The deployment is read from `config/config.yaml` -> `deploy.deployment_id`
(create or update it with `deploy.ipynb`). Override at runtime with `DR_DEPLOYMENT_ID`.


## 1. Connect

Credentials are read (in order) from:
1. `DATAROBOT_ENDPOINT` / `DATAROBOT_API_TOKEN` environment variables, or
2. `~/.config/datarobot/drconfig.yaml`.


In [1]:
import os

import datarobot as dr
import numpy as np
import pandas as pd

print("datarobot SDK:", dr.__version__)

endpoint = os.environ.get("DATAROBOT_ENDPOINT")
token = os.environ.get("DATAROBOT_API_TOKEN")

try:
    if endpoint and token:
        client = dr.Client(endpoint=endpoint, token=token)
    else:
        # falls back to ~/.config/datarobot/drconfig.yaml
        client = dr.Client()
except dr.errors.ClientError as exc:
    raise RuntimeError(
        "DataRobot authentication failed. Refresh your API key at "
        "<app>/account/developer-tools and either update the `token:` line in "
        "~/.config/datarobot/drconfig.yaml or export DATAROBOT_ENDPOINT / "
        "DATAROBOT_API_TOKEN before starting the kernel."
    ) from exc

APP_ROOT = client.endpoint.rsplit("/api/v2", 1)[0]
print("endpoint  :", client.endpoint)
print("app root  :", APP_ROOT)

datarobot SDK: 3.19.0
endpoint  : https://app.jp.datarobot.com/api/v2
app root  : https://app.jp.datarobot.com


## 2. Settings

In [2]:
from src.config import load_config

cfg = load_config("config/config.yaml")

TARGET = cfg["target"]
SMILES_COLUMN = cfg["data"].get("smiles_column", "SMILES")
MODEL_NAME = cfg["model"]["type"].replace("+", "_")

INPUT_DIR = cfg["paths"]["input"]
MODEL_DIR = cfg["paths"]["model"]

predict_cfg = cfg.get("predict") or {}

# What to score: a CSV with a SMILES column. Comes from predict.scoring_csv in
# config/config.yaml; DR_SCORING_CSV overrides at runtime.
SCORING_CSV = os.environ.get("DR_SCORING_CSV") or predict_cfg.get(
    "scoring_csv", os.path.join(INPUT_DIR, "test.csv")
)
# Limit the number of rows scored (predict.row_limit; empty = all).
ROW_LIMIT = predict_cfg.get("row_limit") or None

deploy_cfg = cfg.get("deploy") or {}

# The deployment is identified by its LABEL, which is stable across clusters.
# An id is only ever a shortcut: ids belong to one cluster, so a value cached
# from app.datarobot.com will not resolve on app.jp.datarobot.com (or on-prem),
# and the next cell falls back to the label lookup when that happens.
DEPLOYMENT_LABEL = deploy_cfg.get("model_name") or "smiles_deep_learning_regression"
DEPLOYMENT_ID_HINT = os.environ.get("DR_DEPLOYMENT_ID") or deploy_cfg.get("deployment_id")

print("target          :", TARGET)
print("smiles column   :", SMILES_COLUMN)
print("scoring csv     :", SCORING_CSV)
print("deployment label:", DEPLOYMENT_LABEL)
print("deployment id   :", DEPLOYMENT_ID_HINT or "(none - will look up by label)")

target          : Tc
smiles column   : SMILES
scoring csv     : input/test.csv
deployment label: smiles_deep_learning_regression
deployment id   : 6a6815228635495938d294e0


## 3. Load the deployment

In [3]:
def _resolve_deployment():
    """Find this model's deployment on whichever cluster we are connected to.

    1. Try the cached/overridden id - fast path, and the only path that honours
       DR_DEPLOYMENT_ID when several deployments share a label.
    2. Fall back to looking the label up on this cluster. Ids do not cross
       clusters; the label does.
    """
    if DEPLOYMENT_ID_HINT:
        try:
            d = dr.Deployment.get(DEPLOYMENT_ID_HINT)
            print(f"resolved by id: {d.id}")
            return d
        except dr.errors.ClientError:
            print(
                f"id {DEPLOYMENT_ID_HINT} does not resolve on {APP_ROOT} "
                f"- falling back to label lookup"
            )

    matches = [
        d for d in dr.Deployment.list(search=DEPLOYMENT_LABEL) if d.label == DEPLOYMENT_LABEL
    ]
    if not matches:
        raise RuntimeError(
            f"No deployment labelled {DEPLOYMENT_LABEL!r} on {APP_ROOT}.\n"
            f"Run deploy.ipynb against this cluster first, or export "
            f"DR_DEPLOYMENT_ID to point at an existing one."
        )
    if len(matches) > 1:
        print(
            f"warning: {len(matches)} deployments labelled {DEPLOYMENT_LABEL!r}; "
            f"using the first. Set DR_DEPLOYMENT_ID to disambiguate."
        )
    print(f"resolved by label: {matches[0].id}")
    return matches[0]


deployment = _resolve_deployment()
DEPLOYMENT_ID = deployment.id

model_info = deployment.model or {}
print("label       :", deployment.label)
print("status      :", deployment.status)
print("model       :", model_info.get("type"), "|", model_info.get("id"))
print("target      :", (deployment.model.get("target_name") if deployment.model else None))
print()
print(f"Console -> Deployments:\n  {APP_ROOT}/console-nextgen/deployments/{deployment.id}/overview")

id 6a6815228635495938d294e0 does not resolve on https://app.jp.datarobot.com - falling back to label lookup
resolved by label: 6a8d8b32feab23af000adcd3
label       : smiles_deep_learning_regression
status      : active
model       : smiles_deep_learning_regression | 6a9808664b22d13f4507738e
target      : Tc

Console -> Deployments:
  https://app.jp.datarobot.com/console-nextgen/deployments/6a8d8b32feab23af000adcd3/overview


## 4. Load scoring data

In [4]:
test_df = pd.read_csv(SCORING_CSV)
if ROW_LIMIT:
    test_df = test_df.head(ROW_LIMIT).copy()

if SMILES_COLUMN not in test_df.columns:
    raise RuntimeError(f"{SCORING_CSV} has no `{SMILES_COLUMN}` column: {list(test_df.columns)}")

print(f"rows: {len(test_df)}")
test_df.head()

rows: 131


,SMILES,Tc
0,*CC(*)C(=O)c1ccc(C)cc1,0.197500
1,*CCCCCCCCCNC(=O)C(CCCCCCCCCCCC)C(=O)N*,0.347000
2,*CC(*)CC(C)C,0.208333
3,*CC(*)c1ccc(C(=O)N(C)C)cc1,0.246000
4,*C1C(=O)N(c2ccc(C)cc2)C(=O)C1*,0.140500


## 5. Score through the deployment

`score_pandas` runs a batch prediction job against the deployment and returns the scored frame.

In [5]:
%%time
job, scored = dr.BatchPredictionJob.score_pandas(deployment, test_df[[SMILES_COLUMN]].copy())

try:
    print("batch job:", job.id, "|", job.get_status().get("status"))
except Exception:  # noqa: BLE001
    print("batch job:", job.id)

pred_cols = [c for c in scored.columns if "PREDICTION" in c.upper()]
if not pred_cols:
    raise RuntimeError(f"no prediction column in response: {list(scored.columns)}")

test_df["preds"] = scored[pred_cols[0]].to_numpy()
print("prediction column:", pred_cols[0])
test_df.head()

Streaming DataFrame as CSV data to DataRobot
Created Batch Prediction job ID 6a9809bc29c618820b60e05c
Waiting for DataRobot to start processing
Job has started processing at DataRobot. Streaming results.
batch job: 6a9809bc29c618820b60e05c | COMPLETED
prediction column: Tc_PREDICTION
CPU times: user 667 ms, sys: 43.7 ms, total: 711 ms
Wall time: 1min 11s


,SMILES,Tc,preds
0,*CC(*)C(=O)c1ccc(C)cc1,0.197500,0.204494
1,*CCCCCCCCCNC(=O)C(CCCCCCCCCCCC)C(=O)N*,0.347000,0.345888
2,*CC(*)CC(C)C,0.208333,0.214383
3,*CC(*)c1ccc(C(=O)N(C)C)cc1,0.246000,0.218319
4,*C1C(=O)N(c2ccc(C)cc2)C(=O)C1*,0.140500,0.155593


## 6. Metrics

In [6]:
if TARGET in test_df.columns:
    err = test_df[TARGET].to_numpy() - test_df["preds"].to_numpy()
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    ss_res = (err**2).sum()
    ss_tot = ((test_df[TARGET].to_numpy() - test_df[TARGET].mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
    print(f"rows: {len(test_df)}")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")
    display(test_df[[TARGET, "preds"]].describe())
else:
    print(f"no `{TARGET}` column in the scoring data - predictions only")
    display(test_df[["preds"]].describe())

rows: 131
MAE : 0.0245
RMSE: 0.0346
R2  : 0.8474


,Tc,preds
count,131.000000,131.000000
mean,0.254942,0.257254
std,0.088955,0.085787
min,0.069000,0.074589
25%,0.185667,0.186983
50%,0.235000,0.246931
75%,0.322500,0.335629
max,0.482000,0.508739


## 7. Save

In [7]:
# This notebook is meant to run standalone against an existing deployment,
# so model/ may not exist yet - only train.ipynb creates it, and losing the
# scored frame to a FileNotFoundError after paying for a batch prediction
# job is the worst possible place to fail.
os.makedirs(MODEL_DIR, exist_ok=True)

out_path = os.path.join(MODEL_DIR, f"test_preds_{MODEL_NAME}.csv")
test_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: model/test_preds_dmpnn.csv
